# Notebook 5 — Feature Engineering & Baseline Model (Task 2.1–2.3)

**Why a new notebook, not the EDA notebook:** the EDA notebook (`04_eda_task1.ipynb`) is explicitly read-only analysis — it never re-saves data and is meant to stay a stable, unchanged record of Task 1's findings. This notebook does something structurally different: it transforms the data (new columns, encoding, scaling) and trains a model — a separate deliverable (Task 2.1–2.3) with its own inputs and outputs. Keeping them apart means re-running/debugging the model doesn't risk disturbing the EDA notebook, and vice versa.

**Scope of this notebook:**
- **2.1 Preprocessing** — date-based feature engineering, encoding categorical columns, scaling numeric columns
- **2.2 Pipeline** — a single `sklearn.Pipeline` wrapping preprocessing + a Random Forest Regressor
- **2.3 Loss function** — choosing and defending RMSPE (Root Mean Squared Percentage Error) as the evaluation metric

**Not in scope here** (later notebooks): feature importance / confidence intervals (2.4), model serialization (2.5), the LSTM approach (2.6), MLflow serving (2.7).

**Input:** `store_train.csv` (from Notebook 2). **Output:** a fitted pipeline object kept in memory for this session — saving it to disk is Task 2.5, in the next notebook.

## Step 0 — Setup

In [1]:
# --- Option A: Mount Google Drive ---
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = '/content/drive/MyDrive/rossmann_project/'

# --- Option B: Manual upload ---
# from google.colab import files
# uploaded = files.upload()   # upload store_train.csv
# DATA_DIR = ''

# --- Option C: Local / same directory ---
DATA_DIR = ''

### Logger setup

Same pattern as Notebooks 1–4: a console handler (readable while you work) plus a file handler (`feature_engineering_modeling.log`) that persists every step even after notebook outputs are cleared — this is where Task 2's "log your steps" evidence lives for this notebook.

In [2]:
import pandas as pd
import numpy as np
import logging
import sys
import time

LOG_FILE = 'feature_engineering_modeling.log'
logger = logging.getLogger('feature_engineering_modeling')   # explicit name, not __name__
logger.setLevel(logging.DEBUG)
logger.propagate = False

if not logger.handlers:   # guard against duplicate handlers if this cell is re-run
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')

    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.INFO)
    console_handler.setFormatter(formatter)

    file_handler = logging.FileHandler(LOG_FILE)
    file_handler.setLevel(logging.DEBUG)
    file_handler.setFormatter(formatter)

    logger.addHandler(console_handler)
    logger.addHandler(file_handler)

pd.set_option('display.max_columns', None)
logger.info(f"Libraries imported and logger configured. Full detail is being written to {LOG_FILE}")

2026-09-15 19:22:12,083 - feature_engineering_modeling - INFO - Libraries imported and logger configured. Full detail is being written to feature_engineering_modeling.log


## Step 1 — Load data

In [3]:
store_train = pd.read_csv(DATA_DIR + 'store_train.csv', low_memory=False, parse_dates=['Date'])
store_train['StateHoliday'] = store_train['StateHoliday'].astype(str)   # same safety cast as Notebook 4

logger.info(f"store_train loaded. Shape: {store_train.shape}")
logger.debug(f"store_train dtypes: {dict(store_train.dtypes.astype(str))}")
store_train.head()

2026-09-15 19:22:12,967 - feature_engineering_modeling - INFO - store_train loaded. Shape: (252322, 9)


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555.0,1,1,0,1
1,2,5,2015-07-31,6064,625.0,1,1,0,1
2,3,5,2015-07-31,8314,821.0,1,1,0,1
3,4,5,2015-07-31,13995,1498.0,1,1,0,1
4,5,5,2015-07-31,4822,559.0,1,1,0,1


## Step 2 — Date-based feature engineering (Task 2.1)

**Important ordering decision:** we engineer every feature on the **full** dataset first — including closed-store (`Open == 0`) rows — and only filter down to `Open == 1` at the very end (Step 4), right before modeling.

This matters specifically for the holiday-distance features below. Most stores are *closed* on the holiday itself (`Open == 0`), so if we filtered to `Open == 1` first, the actual holiday dates would mostly disappear from each store's calendar before we ever calculated "days to next holiday" — silently corrupting the feature. Computing it on the full timeline first, then filtering, avoids that bug entirely.

In [4]:
df = store_train.copy()

# --- Basic calendar features ---
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['WeekOfYear'] = df['Date'].dt.isocalendar().week.astype(int)
df['IsWeekend'] = df['DayOfWeek'].isin([6, 7]).astype(int)   # DayOfWeek: 6=Saturday, 7=Sunday

logger.info("Added Year, Month, Day, WeekOfYear, IsWeekend.")

2026-09-15 19:22:13,082 - feature_engineering_modeling - INFO - Added Year, Month, Day, WeekOfYear, IsWeekend.


### Beginning / mid / end of month

In [5]:
df['MonthPeriod'] = pd.cut(df['Day'], bins=[0, 10, 20, 31], labels=['Start', 'Mid', 'End'])
logger.info(f"MonthPeriod distribution: {df['MonthPeriod'].value_counts().to_dict()}")

2026-09-15 19:22:13,124 - feature_engineering_modeling - INFO - MonthPeriod distribution: {'End': 90565, 'Mid': 83707, 'Start': 78050}


### Days to next holiday / days since last holiday

Computed **per store** (each store has its own trading calendar) using a vectorized `searchsorted` lookup rather than a slow row-by-row loop — this runs in under a second even across 1,115 stores and ~1M rows.

We use `999` as a sentinel for "no holiday found nearby in this store's data" (e.g. the very first days of the dataset, before any holiday has occurred yet). We then **clip both features to a maximum of 90 days** — beyond ~3 months, the exact distance stops being meaningful ("far from a holiday" is far from a holiday whether it's 200 or 999 days), and leaving the raw sentinel in would badly distort `StandardScaler`'s mean/std later, since a few extreme 999s would dominate the scale.

In [6]:
t0 = time.time()

df = df.sort_values(['Store', 'Date']).reset_index(drop=True)
df['IsHoliday'] = (df['StateHoliday'] != '0').astype(int)
df['_DateOrd'] = df['Date'].map(pd.Timestamp.toordinal)

next_list, prev_list = [], []
for store_id, g in df.groupby('Store', sort=False):
    holiday_ords = g.loc[g['IsHoliday'] == 1, '_DateOrd'].sort_values().unique()
    dates = g['_DateOrd'].values
    if len(holiday_ords) == 0:
        next_list.append(np.full(len(dates), 999))
        prev_list.append(np.full(len(dates), 999))
        continue
    idx_next = np.searchsorted(holiday_ords, dates, side='left')
    idx_prev = idx_next - 1
    next_vals = np.where(idx_next < len(holiday_ords),
                          holiday_ords[np.clip(idx_next, 0, len(holiday_ords) - 1)] - dates, 999)
    prev_vals = np.where(idx_prev >= 0,
                          dates - holiday_ords[np.clip(idx_prev, 0, len(holiday_ords) - 1)], 999)
    next_list.append(next_vals)
    prev_list.append(prev_vals)

df['DaysToNextHoliday'] = np.concatenate(next_list)
df['DaysSinceLastHoliday'] = np.concatenate(prev_list)

# Clip the sentinel so it doesn't distort scaling later
df['DaysToNextHoliday'] = df['DaysToNextHoliday'].clip(upper=90)
df['DaysSinceLastHoliday'] = df['DaysSinceLastHoliday'].clip(upper=90)

df = df.drop(columns=['_DateOrd'])

logger.info(f"Holiday-distance features computed in {time.time()-t0:.2f}s")
logger.debug(f"DaysToNextHoliday describe: {df['DaysToNextHoliday'].describe().to_dict()}")

2026-09-15 19:22:15,074 - feature_engineering_modeling - INFO - Holiday-distance features computed in 1.93s


### Extra feature — `IsPromoMonth`

`Promo2` stores run their recurring promotion only in specific months, listed in `PromoInterval` (e.g. `"Jan,Apr,Jul,Oct"`). This flag checks whether the **current row's month** falls inside that store's active promo months — a much more usable signal for a model than the raw `PromoInterval` string.

**Data quirk worth knowing:** one of the four interval groups in this dataset is literally `"Mar,Jun,Sept,Dec"` — note **`Sept`**, not the 3-letter `Sep` used everywhere else. An exact-match approach (splitting the string and comparing `'Sep' == 'Sept'`) silently misses September as a promo month for every store in that group — we caught this by testing the two approaches against each other and finding a mismatch. Using **substring matching** (`'Sep' in 'Sept'`) instead handles it correctly and is also ~3x faster than a row-by-row `.apply()`.

In [7]:
month_map = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
df['MonthAbbr'] = df['Month'].map(month_map)

mask = np.zeros(len(df), dtype=bool)
for abbr in month_map.values():
    sel = (df['MonthAbbr'] == abbr) & (df['PromoInterval'].str.contains(abbr, na=False))
    mask |= sel
df['IsPromoMonth'] = ((df['Promo2'] == 1) & mask).astype(int)

df = df.drop(columns=['MonthAbbr'])
logger.info(f"IsPromoMonth: {df['IsPromoMonth'].sum():,} rows flagged "
            f"({df['IsPromoMonth'].mean()*100:.1f}% of all rows)")

KeyError: 'PromoInterval'

### Extra features — turning static competition/promo dates into a usable duration

`CompetitionOpenSinceMonth/Year` and `Promo2SinceWeek/Year` are calendar dates — not directly useful to a model as raw numbers (a model shouldn't treat the *year* 2010 as "bigger" and therefore "more" than 2008 in a meaningful way here). Converting them into **"how many months/weeks has this been running as of this row's date"** gives the model an actual duration it can reason about.

For stores where we never learned the competitor's opening date (`CompetitionOpenSinceKnown == 0`, filled with 0/0 back in Notebook 1), we set the duration to `0` rather than computing a nonsense negative/huge number from the placeholder date.

In [ ]:
df['CompetitionOpenMonths'] = ((df['Year'] - df['CompetitionOpenSinceYear']) * 12 +
                                (df['Month'] - df['CompetitionOpenSinceMonth']))
df.loc[df['CompetitionOpenSinceKnown'] == 0, 'CompetitionOpenMonths'] = 0
df['CompetitionOpenMonths'] = df['CompetitionOpenMonths'].clip(lower=0, upper=240)  # cap at 20 years

df['Promo2OpenWeeks'] = ((df['Year'] - df['Promo2SinceYear']) * 52 +
                          (df['WeekOfYear'] - df['Promo2SinceWeek']))
df.loc[df['Promo2'] == 0, 'Promo2OpenWeeks'] = 0
df['Promo2OpenWeeks'] = df['Promo2OpenWeeks'].clip(lower=0)

logger.info("Added CompetitionOpenMonths and Promo2OpenWeeks.")
logger.debug(f"CompetitionOpenMonths describe: {df['CompetitionOpenMonths'].describe().to_dict()}")

## Step 3 — Encoding & scaling plan (Task 2.1)

| Column | Type | Treatment | Why |
|---|---|---|---|
| `StateHoliday`, `StoreType`, `Assortment`, `MonthPeriod` | Nominal categorical | **One-hot encode** | No natural order between categories — one-hot avoids implying a false ranking |
| `Store`, `DayOfWeek`, `Promo`, `SchoolHoliday`, `Promo2`, `IsWeekend`, `IsPromoMonth`, `Year`, `Month`, `Day`, `WeekOfYear`, `CompetitionDistance`, `CompetitionOpenSinceKnown`, `CompetitionOpenMonths`, `Promo2OpenWeeks`, `DaysToNextHoliday`, `DaysSinceLastHoliday` | Numeric | **Standard-scale** | Puts every numeric feature on a comparable scale (mean 0, unit variance) — required for distance-based algorithms, and harmless for the tree-based model we use here |
| `Date`, `PromoInterval`, `CompetitionOpenSinceMonth/Year`, `Promo2SinceWeek/Year` | Raw / superseded | **Drop** | Either not model-usable directly (`Date`, `PromoInterval`) or already captured by a derived feature above |
| `Sales`, `Customers` | Target / leakage | **Not a feature** | `Sales` is the target. `Customers` is only known *after* a sale happens — it would leak the answer if used as an input feature for forecasting future days |

> **Why `Customers` is dropped as a feature:** at prediction time (6 weeks ahead), you won't know how many customers will show up — that's arguably as hard to know in advance as `Sales` itself. Using it as a training feature would make the model look artificially accurate, then fail in real deployment where `Customers` isn't available. This is a classic **data leakage** trap worth calling out explicitly.

In [ ]:
NUMERIC_FEATURES = [
    'Store', 'DayOfWeek', 'Promo', 'SchoolHoliday', 'Promo2', 'IsWeekend', 'IsPromoMonth',
    'Year', 'Month', 'Day', 'WeekOfYear',
    'CompetitionDistance', 'CompetitionOpenSinceKnown', 'CompetitionOpenMonths', 'Promo2OpenWeeks',
    'DaysToNextHoliday', 'DaysSinceLastHoliday'
]
CATEGORICAL_FEATURES = ['StateHoliday', 'StoreType', 'Assortment', 'MonthPeriod']
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET = 'Sales'

logger.info(f"{len(NUMERIC_FEATURES)} numeric features, {len(CATEGORICAL_FEATURES)} categorical features.")

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), NUMERIC_FEATURES),
    ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL_FEATURES)
])
logger.info("ColumnTransformer built: StandardScaler for numeric, OneHotEncoder for categorical.")

## Step 4 — Filter to Open==1, then a time-based train/validation split

**Filtering to `Open == 1`:** closed-store days always have `Sales == 0` by definition — that's not something a model needs to "learn" from features, it's a trivial rule (`Open == 0` -> `Sales = 0`). Training on those rows mostly just teaches the model to predict near-zero whenever `Open == 0`, diluting the learning signal for the actual behaviour we care about. The standard, well-documented approach for this dataset -- and the one we use here -- is: **train only on `Open == 1` rows, and apply the `Open == 0` -> `Sales = 0` rule directly at serving time** (Task 3), bypassing the model entirely for closed days.

**Time-based split, not random:** the real task is forecasting **6 weeks ahead** — a random shuffle-split would let the model "peek" at rows from dates surrounding a validation row, which is impossible in real deployment (the future doesn't exist yet when you're forecasting it). We instead hold out the **last 6 weeks of the training window** as validation, mirroring the actual test set's structure (`store_test` covers Aug-Sep 2015, also roughly 6-7 weeks).

In [ ]:
model_df = df[df['Open'] == 1].copy()
logger.info(f"Filtered to Open==1: {len(model_df):,} rows (from {len(df):,} total, "
            f"{len(df)-len(model_df):,} closed-store rows excluded).")

# Diagnostic: always log the actual date range you're splitting -- if this looks wrong
# (e.g. spans only a few weeks instead of ~2.5 years), df is stale or store_train.csv is
# truncated. Restart the runtime and re-run Steps 1-2 from the top before continuing.
logger.info(f"model_df date range: {model_df['Date'].min().date()} to {model_df['Date'].max().date()} "
            f"({(model_df['Date'].max() - model_df['Date'].min()).days} days)")

max_date = model_df['Date'].max()
val_cutoff = max_date - pd.Timedelta(days=42)   # last 6 weeks

train_df = model_df[model_df['Date'] <= val_cutoff]
val_df = model_df[model_df['Date'] > val_cutoff]

# Fail fast, right here, with a precise message -- instead of letting an empty train_df
# silently reach the pipeline and surface as a cryptic ValueError inside StandardScaler.
if len(train_df) == 0 or len(val_df) == 0:
    logger.error(f"Empty split! train_df={len(train_df)} rows, val_df={len(val_df)} rows. "
                 f"model_df spans {model_df['Date'].min().date()} to {model_df['Date'].max().date()}, "
                 f"val_cutoff={val_cutoff.date()}.")
    raise ValueError(
        f"Time-based split produced an empty set (train={len(train_df)}, val={len(val_df)} rows).\n"
        f"model_df date range was only {model_df['Date'].min().date()} to {model_df['Date'].max().date()}.\n"
        "This almost always means either:\n"
        "  1) 'df' is stale from an earlier partial run -- restart the runtime and re-run "
        "Steps 1-2 from the top, or\n"
        "  2) store_train.csv itself is truncated/wrong -- re-check the file loaded in Step 1."
    )

logger.info(f"Train: {len(train_df):,} rows, up to {val_cutoff.date()}")
logger.info(f"Validation: {len(val_df):,} rows, {val_df['Date'].min().date()} to {val_df['Date'].max().date()}")

X_train, y_train = train_df[ALL_FEATURES], train_df[TARGET]
X_val, y_val = val_df[ALL_FEATURES], val_df[TARGET]

## Step 5 — Build the pipeline with a Random Forest Regressor (Task 2.2)

We wrap the `RandomForestRegressor` in a `TransformedTargetRegressor` that applies `log1p` to `Sales` before training and `expm1` back to actual Sales units on prediction -- handled automatically, so `pipeline.predict()` always returns real Sales values, never log-values.

**Why log-transform the target:** the EDA notebook found an **8x gap** between the top- and bottom-selling stores. Training directly on raw `Sales` means the model's errors on a 20,000/day store and a 3,000/day store get treated with equal absolute weight -- but a 500-unit miss is trivial for the first store and huge for the second. `log1p` compresses that scale difference, so the model -- and the loss it's optimized against -- treats proportional errors more fairly across very differently sized stores.

**Why Random Forest:** it's a strong, low-maintenance baseline for tabular regression -- handles non-linear interactions (e.g. `Promo x StoreType`) without manual feature crosses, doesn't require feature scaling to function correctly (we still scale for consistency/future algorithm swaps), and gives free feature-importance output for Task 2.4.

> **Runtime note (measured, not estimated):** `n_estimators=50, max_depth=15` was timed at **~3.5 minutes** end-to-end on the full ~804K-row training set. Doubling to `n_estimators=100, max_depth=20` roughly doubles that to ~7 minutes for a modest accuracy gain -- worth trying once you've confirmed the pipeline runs, but the settings below are the tested default so you know exactly what to expect the first time you run this.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import Pipeline

rf_model = TransformedTargetRegressor(
    regressor=RandomForestRegressor(
        n_estimators=50,
        max_depth=15,
        min_samples_leaf=5,
        n_jobs=-1,
        random_state=42
    ),
    func=np.log1p,
    inverse_func=np.expm1
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', rf_model)
])

logger.info("Pipeline assembled: ColumnTransformer -> TransformedTargetRegressor(RandomForestRegressor).")
pipeline

### Fit the pipeline

In [ ]:
t0 = time.time()
pipeline.fit(X_train, y_train)
fit_seconds = time.time() - t0
logger.info(f"Pipeline fit complete in {fit_seconds/60:.1f} minutes on {len(X_train):,} rows.")

## Step 6 — Choose and defend a loss function (Task 2.3)

### Chosen metric: RMSPE (Root Mean Squared Percentage Error)

$$\text{RMSPE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}\left(\frac{y_i - \hat{y}_i}{y_i}\right)^2}$$

**Why this, specifically, over plain RMSE or MAE:**

1. **It's directly comparable across stores of very different sizes.** The EDA notebook found an 8x gap between the top- and bottom-selling stores. A flat RMSE of 500 means something completely different for a 2,700/day store (18.5% error) versus a 21,700/day store (2.3% error). RMSPE reports **relative** error, so the finance team gets one number that means the same thing everywhere -- no need to mentally re-scale it store by store.
2. **It's directly interpretable to a non-technical audience.** "The model is typically off by about X%" is immediately meaningful to the finance team, store managers, and stakeholders who aren't data scientists -- unlike a raw RMSE number in currency units, which requires knowing typical sales volume to judge whether it's good or bad.
3. **It's the actual metric this real-world competition was judged on.** This isn't an arbitrary choice -- RMSPE is the official evaluation metric for the original Rossmann Store Sales competition this dataset comes from, so using it here keeps our evaluation aligned with how this exact problem is conventionally judged, and makes any published benchmark numbers directly comparable to ours.

**Trade-off worth naming:** RMSPE is undefined when `Sales == 0` (division by zero) -- not an issue for us since we already excluded closed-store (`Sales == 0`) rows from training and validation. It can also be disproportionately sensitive to error on very *low*-sales days (a 50-unit miss on a 100-unit day is 50%, the same 50-unit miss on a 10,000-unit day is 0.5%) -- worth keeping in mind when interpreting the number rather than treating it as the single, final word on model quality. We report plain MAE and RMSE alongside it for that reason.

`sklearn` has no built-in RMSPE, so we implement it directly.

In [ ]:
def rmspe(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mask = y_true != 0   # guard against division by zero (shouldn't trigger here, but safe regardless)
    return np.sqrt(np.mean(((y_true[mask] - y_pred[mask]) / y_true[mask]) ** 2))

### Evaluate on the held-out validation set (last 6 weeks)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

val_preds = pipeline.predict(X_val)

val_rmspe = rmspe(y_val.values, val_preds)
val_mae = mean_absolute_error(y_val, val_preds)
val_rmse = np.sqrt(mean_squared_error(y_val, val_preds))

logger.info(f"Validation RMSPE: {val_rmspe*100:.2f}%")
logger.info(f"Validation MAE:   {val_mae:.1f}")
logger.info(f"Validation RMSE:  {val_rmse:.1f}")

print(f"RMSPE: {val_rmspe*100:.2f}%")
print(f"MAE:   {val_mae:.1f}")
print(f"RMSE:  {val_rmse:.1f}")

---
## What's next

This notebook covered Task 2.1 (preprocessing), 2.2 (pipeline), and 2.3 (loss function + evaluation). Coming next, in a following notebook:
- **2.4** -- feature importance from the fitted Random Forest, and a confidence-interval estimate for predictions
- **2.5** -- serializing this pipeline with a timestamped filename for tracking daily prediction runs
- **2.6 / 2.7** -- the LSTM deep-learning approach and MLflow-served inference